# Biscuit — Candidate Ranker Sandbox

This notebook runs the Biscuit candidate ranking system end-to-end on a 50-candidate sample.
It demonstrates the full pipeline: load → filter honeypots → score → rank → output CSV.

**Runtime:** < 5 seconds on the 50-candidate sample (well within the 5-minute CPU budget).

### Step 1: Clone the repository

In [ ]:
!git clone https://github.com/vinayaksonthalia/biscuit-candidate-ranker.git
%cd biscuit-candidate-ranker

### Step 2: Run the ranker on 50 sample candidates

Uses `--top-n 50` since the sample dataset contains only 50 candidates.
The full pipeline processes 100K candidates well within the 5-minute CPU budget.

In [ ]:
!python rank.py --candidates data/sample_candidates.json --out sandbox_output.csv --top-n 50

### Step 3: Verify output format

In [ ]:
import csv

with open('sandbox_output.csv') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

# Format checks
assert reader.fieldnames == ['candidate_id', 'rank', 'score', 'reasoning'], f'Bad columns: {reader.fieldnames}'
assert len(rows) == 50, f'Expected 50 rows, got {len(rows)}'
ranks = [int(r['rank']) for r in rows]
assert ranks == list(range(1, 51)), 'Ranks not sequential 1-50'
scores = [float(r['score']) for r in rows]
assert all(scores[i] >= scores[i+1] for i in range(len(scores)-1)), 'Scores not monotonically non-increasing'
assert all(r['candidate_id'].startswith('CAND_') for r in rows), 'Invalid candidate IDs'
assert len(set(r['candidate_id'] for r in rows)) == 50, 'Duplicate candidate IDs'
assert all(r['reasoning'].strip() for r in rows), 'Empty reasonings found'

print('All format checks passed!')
print(f'  Rows: {len(rows)}')
print(f'  Score range: {scores[-1]:.4f} — {scores[0]:.4f}')
print(f'  All reasonings non-empty: True')
print(f'  Unique reasonings: {len(set(r["reasoning"] for r in rows))}')

### Step 4: Display ranked results

In [ ]:
import csv
from html import escape
from IPython.display import HTML, display

with open('sandbox_output.csv') as f:
    rows = list(csv.DictReader(f))[:10]

headers = ['candidate_id', 'rank', 'score', 'reasoning']
html = '<table><thead><tr>' + ''.join(f'<th>{escape(h)}</th>' for h in headers) + '</tr></thead><tbody>'
for row in rows:
    html += '<tr>' + ''.join(f'<td>{escape(row[h])}</td>' for h in headers) + '</tr>'
html += '</tbody></table>'
display(HTML(html))

### Step 5: Verify determinism (run twice, compare)

In [ ]:
import subprocess, filecmp

subprocess.run(['python', 'rank.py', '--candidates', 'data/sample_candidates.json',
                '--out', 'sandbox_output_2.csv', '--top-n', '50'], check=True)

if filecmp.cmp('sandbox_output.csv', 'sandbox_output_2.csv'):
    print('Determinism check PASSED — identical output across runs.')
else:
    print('FAILED — outputs differ between runs!')